# Tutorial SRA LLM (versão TinyShakespeare)

Neste notebook, você experimentará as etapas para treinar um modelo de linguagem pequena (Tiny LLM) do zero usando SRA (Synaptic Routing Architecture).
Usamos o texto da peça de Shakespeare (TinyShakespeare) como conjunto de dados e visualizamos como o SRA gera texto e que tipo de roteamento (uso de sinapse) ele executa após o aprendizado.

## 1. Configuração do ambiente
Adicione a instalação da biblioteca necessária e o caminho para o diretório `src` onde o modelo SRA está definido.

In [ ]:
# Colab環境でのみ実行してください（ローカル環境の場合はスキップ可）
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch tiktoken matplotlib seaborn requests

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

## 2. Preparação e tokenização do conjunto de dados
Baixe o conjunto de dados TinyShakespeare e tokenize-o usando o `tiktoken` da OpenAI.

In [ ]:
import requests
import torch
import tiktoken

# TinyShakespeareデータのダウンロード
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text

print(f"テキストの長さ: {len(text)} 文字")
print("サンプル:\n", text[:100])

# トークナイザーの準備
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

# テキスト全体をエンコード（時間がかかるため最初の一部のみ使用することも可能です）
# ここでは学習を早く回すため、最初の10万トークン程度に絞ります
tokens = tokenizer.encode(text[:500000], allowed_special="all")
data = torch.tensor(tokens, dtype=torch.long)
print(f"トークン数: {len(data)}")

## 3. Construindo o modelo SRA LLM
Chama `MoESRALanguageModel` implementado em `src/sra_language_models.py`.
Desta vez, usaremos configurações extremamente pequenas (`dim=128`, `layers=2`) para que possamos verificar o aprendizado em pouco tempo.

In [ ]:
from sra_language_models import MoESRALanguageModel

# ミニマムなモデル設定
dim = 128
layers = 2
num_synapses = 16  # ルーティング先の専門家（シナプス）の数
k = 2              # 1トークンあたり同時に選ばれるシナプスの数
syn_hidden = 256
max_seq_len = 64   # 一度に扱う系列長

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"使用デバイス: {device}")

model = MoESRALanguageModel(
    vocab_size=vocab_size,
    dim=dim,
    layers=layers,
    num_synapses=num_synapses,
    k=k,
    syn_hidden=syn_hidden,
    max_seq_len=max_seq_len
).to(device)

print(f"総パラメータ数: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

## 4. Ciclo de pré-treinamento
Fornecemos uma função que gera um minilote simples e gira o aprendizado em poucos minutos. Ao adicionar a "perda de equilíbrio de carga" exclusiva do SRA, incentivamos que todas as sinapses sejam usadas igualmente.

In [ ]:
import random
import torch.nn.functional as F
from sra_experiment import make_optimizer, load_balance_loss

def get_batch(data, batch_size, seq_len):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    
    max_start = len(data) - seq_len - 1
    for i in range(batch_size):
        start = random.randint(0, max_start)
        x[i] = data[start:start+seq_len]
        y[i] = data[start+1:start+seq_len+1]
        
    return x.to(device), y.to(device)

# 学習パラメータ
batch_size = 32
steps = 300  # 学習ステップ数（短めに設定）
lr = 5e-4
opt = make_optimizer(model, lr)

model.train()
for step in range(1, steps + 1):
    x, y = get_batch(data, batch_size, max_seq_len)
    
    # モデルの推論とルーティング結果の取得
    logits, router_logits = model(x, dense=False)
    
    # 損失の計算（言語モデルのクロスエントロピー + SRAのロードバランス）
    ce_loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    lb_loss = load_balance_loss(router_logits)
    loss = ce_loss + 0.1 * lb_loss
    
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    
    if step % 50 == 0:
        print(f"Step {step}/{steps} | Loss: {loss.item():.4f} (CE: {ce_loss.item():.4f}, LB: {lb_loss.item():.4f})")

## 5. Geração de texto e visualização de roteamento
Gere texto usando o modelo aprendido e use um mapa de calor para visualizar por qual sinapse cada token passou durante o processo de geração.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
prompt_text = "ROMEO:"
prompt_tokens = tokenizer.encode(prompt_text)

generated_tokens = prompt_tokens.copy()
all_routing_probs = []

print(f"Prompt: {prompt_text}")

# 30トークン生成
with torch.no_grad():
    for _ in range(30):
        x = torch.tensor([generated_tokens[-max_seq_len:]], device=device)
        logits, router_logits = model(x)
        
        # 最後のトークンの予測ロジット
        next_token_logits = logits[0, -1, :]
        probs = torch.softmax(next_token_logits / 0.8, dim=-1) # 温度付き
        next_token = torch.multinomial(probs, num_samples=1).item()
        generated_tokens.append(next_token)
        
        # 最後の層のルーティング確率を取得 (トークン x シナプス)
        last_layer_logits = router_logits[-1] # shape: (1, seq_len, num_synapses)
        last_token_routing = torch.softmax(last_layer_logits[0, -1, :], dim=-1)
        all_routing_probs.append(last_token_routing.cpu().numpy())

generated_text = tokenizer.decode(generated_tokens)
print(f"Generated Text:\n{generated_text}")

# 可視化
import numpy as np
routing_matrix = np.array(all_routing_probs)  # (生成トークン数, シナプス数)
generated_str_tokens = [tokenizer.decode([t]).replace('\n', '\\n') for t in generated_tokens[len(prompt_tokens):]]

plt.figure(figsize=(10, 6))
sns.heatmap(routing_matrix.T, cmap="viridis", xticklabels=generated_str_tokens)
plt.title("Routing Probabilities per Token (Last Layer)")
plt.xlabel("Generated Token")
plt.ylabel("Synapse ID")
plt.show()